In [1]:
from sympy import symbols, Rational, MutableDenseNDimArray, Function, diff,Add
from sympy.utilities.codegen import codegen
from sympy.codegen.rewriting import optimize, optims_c99
from sympy.simplify.cse_main import cse
import sympy as sp
from sympy import S,I,re,im,Matrix,sqrt,simplify,det,KroneckerDelta
from sympy.printing.c import C99CodePrinter, Assignment
from sympy import init_printing
from sympy.functions.special.tensor_functions import LeviCivita
from codegen.codegen_utils import *

init_printing()
printer=MyPrinter()

### Coordinates and symbolic expressions of geometric quantities

We introduce spatial coordinates

$(x^1,x^2,x^3) = (x,y,z)\,,$

and work in three dimensions.

We define the conformal spatial metric $\tilde\gamma_{ij}$ with six independent components,

$\tilde\gamma_{ij}=\begin{pmatrix}
\tilde\gamma_{xx} & \tilde\gamma_{xy} & \tilde\gamma_{xz} \\
\tilde\gamma_{xy} & \tilde\gamma_{yy} & \tilde\gamma_{yz} \\
\tilde\gamma_{xz} & \tilde\gamma_{yz} & \tilde\gamma_{zz}
\end{pmatrix}\,,$

where each component is an arbitrary function of $(x,y,z)$.

Its determinant is denoted by

$\tilde\gamma \equiv \det(\tilde\gamma_{ij})\,.$

Assuming the conformal constraint $\tilde\gamma = 1$, the inverse conformal metric is constructed as

$\tilde\gamma^{ij} = \tilde\gamma\,(\tilde\gamma^{-1})^{ij}\,.$


We introduce the conformal factor $\chi(x,y,z)$ and define the physical spatial metric and its inverse as

$\gamma_{ij} = \frac{\tilde\gamma_{ij}}{\chi}, \qquad \gamma^{ij} = \chi\,\tilde\gamma^{ij}\,.$

The determinant of the physical metric then satisfies $\det(\gamma_{ij}) = \chi^{-3}$, using $\det(\tilde\gamma_{ij}) = 1$.

The trace-free conformal extrinsic curvature $\tilde A_{ij}$ and the trace $\hat K$ are introduced as independent fields.
The physical extrinsic curvature is reconstructed via

$K_{ij}=\frac{1}{\chi}\left(\tilde A_{ij}+\frac{1}{3}\hat K\,\tilde\gamma_{ij}\right)\,.$

The Christoffel symbols of the physical metric are split into a conformal part and a conformal-factor correction:

$\Gamma^i{}_{jk}=\tilde\Gamma^i{}_{jk}+C^i{}_{jk}\,.$

The conformal Christoffels are

$\tilde\Gamma^i{}_{jk}=\frac{1}{2}\tilde\gamma^{il}\left(\partial_j \tilde\gamma_{lk}+\partial_k \tilde\gamma_{lj}-\partial_l \tilde\gamma_{jk}\right)\,,$

while the correction due to $\chi$ is

$C^i{}_{jk}=-\frac{1}{2\chi}\left(\delta^i{}_k\,\partial_j\chi+\delta^i{}_j\,\partial_k\chi-\tilde\gamma_{jk}\tilde\gamma^{il}\partial_l\chi\right)\,.$

Using $\Gamma^i{}_{jk}$, the spatial Riemann tensor is constructed as

$R^i{}_{jkl}=\partial_k \Gamma^i{}_{lj}-\partial_l \Gamma^i{}_{kj}+\Gamma^i{}_{km}\Gamma^m{}_{lj}-\Gamma^i{}_{lm}\Gamma^m{}_{kj}\,.$

The Ricci tensor is obtained by contraction (no metric needed):
$R_{ij} = R^k{}_{ikj}\,.$

Overall, the code symbolically constructs $\gamma_{ij}$, $K_{ij}$, $\Gamma^i{}_{jk}$, $R^i{}_{jkl}$, and $R_{ij}$ in a conformal 3-metric formulation suitable for BSSN-type calculations and curvature-based diagnostics.


In [2]:
# spatial coordinates
xu=sp.symbols('x y z',real=True)
dim=len(xu)

# definition of conformal metric, 6 independent components are real functions of coordinates
gtxx,gtxy,gtxz,gtyy,gtyz,gtzz=[Function(f"gtdd[{i}]", real=True)(*xu) for i in range(6)]
gtdd=Matrix([
    [gtxx,gtxy,gtxz],
    [gtxy,gtyy,gtyz],
    [gtxz,gtyz,gtzz]
])

# determinant
detgt = simplify(gtdd.det())

# inverse metric assuming unit determinant 
gtuu=simplify(detgt*gtdd.inv())

# conformal factor
chi=Function("chi",real=True)(*xu)

# non-conformal metric
gdd=gtdd/chi

# inverse
guu=gtuu*chi

# determinant: assuming explicitly det(gtdd)=1
detg=simplify(gdd.det()/detgt)

# trace-free conformal extrinsic curvature
Atxx,Atxy,Atxz,Atyy,Atyz,Atzz=[Function(f"Atdd[{i}]", real=True)(*xu) for i in range(6)]
Atdd=Matrix([
    [Atxx,Atxy,Atxz],
    [Atxy,Atyy,Atyz],
    [Atxz,Atyz,Atzz]
])

# extrinsic curvature trace
Khat=Function("Khat",real=True)(*xu)

# extrinsic curvature
Kdd=simplify((Atdd+Rational(1,3)*Khat*gtdd)/chi)

# Christoffels: Gammaudd=Gammatudd+Cudd
Gammatudd=sp.MutableDenseNDimArray.zeros(dim,dim,dim)
Cudd=sp.MutableDenseNDimArray.zeros(dim,dim,dim)
Gammaudd=sp.MutableDenseNDimArray.zeros(dim,dim,dim)
for i in range(dim):
    for j in range(dim):
        for k in range(dim):
            for l in range(dim):
                Gammatudd[i,j,k]+=Rational(1,2)*gtuu[i,l]*(diff(gtdd[l,k],xu[j])+diff(gtdd[l,j],xu[k])-diff(gtdd[j,k],xu[l]))
                Cudd[i,j,k]+=-Rational(1,2)/chi*(KroneckerDelta(i,k)* diff(chi,xu[j])+KroneckerDelta(j,k)* diff(chi,xu[i])-gtdd[i,j]*gtuu[k,l]*diff(chi,xu[l]))
            Gammaudd[i,j,k]=simplify(Gammatudd[i,j,k]+Cudd[i,j,k])

# Riemann
Ruddd=sp.MutableDenseNDimArray.zeros(dim, dim, dim, dim)
GGuddd=sp.MutableDenseNDimArray.zeros(dim, dim, dim, dim)
for i in range(dim):
    for j in range(dim):
        for k in range(dim):
            for l in range(dim):
                for m in range(dim):
                    GGuddd[i,j,k,l] += Gammaudd[i,j,m]*Gammaudd[m,k,l]
                Ruddd[i,j,k,l]=diff(Gammaudd[i,l,j],xu[k])-diff(Gammaudd[i,k,j], xu[l])+GGuddd[i,k,l,j]-GGuddd[i,l,k,j]

# Ricci tensor
Rdd=sp.MutableDenseNDimArray.zeros(dim, dim)
for i in range(dim):
    for j in range(dim):
        for k in range(dim):
            Rdd[i,j]+=Ruddd[k,i,k,j]

### Tetrad construction

We define three linearly independent spatial vectors with contravariant components

$
(v_r)^i = (x_1,x_2,x_3),
\qquad
(v_\phi)^i = (-x_2,x_1,0),
$

and define the third direction by

$
(v_\theta)^i
= \sqrt{\gamma}\,\gamma^{il}\epsilon_{ljk}\,(v_\phi)^j\,(v_r)^k ,
$

where $\gamma \equiv \det(\gamma_{ij})$ and $\epsilon_{ijk}$ is the Levi--Civita symbol associated with the spatial metric $\gamma_{ij}$.

From the vectors $\{\mathbf v_\phi,\mathbf v_r,\mathbf v_\theta\}$ we construct an orthonormal basis
$\{\mathbf e_\phi,\mathbf e_r,\mathbf e_\theta\}$ using the Gram--Schmidt procedure with respect to the spatial metric $\gamma_{ij}$,
with inner product

$
\langle \mathbf u,\mathbf v\rangle = u^i \gamma_{ij} v^j .
$

First, the azimuthal direction is normalized:

$
\Omega_{11} = \langle \mathbf v_\phi,\mathbf v_\phi\rangle, \qquad
\mathbf e_\phi = \frac{\mathbf v_\phi}{\sqrt{\Omega_{11}}} .
$

Next, the radial direction is orthogonalized with respect to $\mathbf e_\phi$ and normalized:

$
\Omega_{12} = \langle e_\phi,\mathbf v_r\rangle, \qquad
\mathbf w_r = \mathbf v_r - \Omega_{12}\,\mathbf e_\phi ,
$

$
\Omega_{22} = \langle \mathbf w_r,\mathbf w_r\rangle, \qquad
\mathbf e_r = \frac{\mathbf w_r}{\sqrt{\Omega_{22}}} .
$

Finally, the polar direction is orthogonalized with respect to both previous directions and normalized:

$
\Omega_{13} = \langle \mathbf e_\phi,\mathbf v_\theta\rangle, \qquad
\Omega_{23} = \langle \mathbf e_r,\mathbf v_\theta\rangle ,
$

$
\mathbf w_\theta= \mathbf v_\theta- \Omega_{13}\,\mathbf e_\phi- \Omega_{23}\,\mathbf e_r ,
$

$
\Omega_{33} = \langle \mathbf w_\theta,\mathbf w_\theta\rangle, \qquad
\mathbf e_\theta = \frac{\mathbf w_\theta}{\sqrt{\Omega_{33}}} .
$

The resulting basis is orthonormal with respect to $g_{ij}$.


In [11]:
# coordinates
x1,x2,x3=symbols('xx1 xx2 xx3',real=True)

# azimuthal vector 
vecAzimuthal=Matrix([-x2,x1,0])

# radial vector
vecRadial=Matrix([x1,x2,x3])

# constructing polar vector
vecPolar=sp.zeros(dim,1)
for i in range(dim):
    for j in range(dim):
        for k in range(dim):
            for l in range(dim):
                vecPolar[i]+=simplify(sqrt(detg)*guu[i,l]*LeviCivita(l,j,k)*vecAzimuthal[j]*vecRadial[k])

# orthonormal basis from Gram-Schmidt procedure
# phi-direction
wAzimuthal=vecAzimuthal
Omega11=(wAzimuthal.T*gdd*wAzimuthal)[0]
eAzimuthal=simplify(wAzimuthal/sqrt(Omega11))
# r-direction
Omega12=(eAzimuthal.T*gdd*vecRadial)[0]
wRadial=vecRadial-Omega12*eAzimuthal
Omega22=(wRadial.T*gdd*wRadial)[0]
eRadial=simplify(wRadial/sqrt(Omega22))
# theta-direction
Omega13=simplify((eAzimuthal.T*gdd*vecPolar)[0])
Omega23=simplify((eRadial.T*gdd*vecPolar)[0])
wPolar=simplify(vecPolar-Omega13*eAzimuthal-Omega23*eRadial)
Omega33=simplify((wPolar.T*gdd*wPolar)[0])
ePolar=simplify(wPolar/sqrt(Omega33))
# check orthogonality
ortho1=simplify((eAzimuthal.T*gdd*eRadial)[0])
ortho2=simplify((eAzimuthal.T*gdd*ePolar)[0])
ortho3=simplify((eRadial.T*gdd*ePolar)[0])
print(ortho1,ortho3,ortho3)

# tetrad (l,n,m,mbar), only n and mbar are needed for psi4
# n
nXre,nYre,nZre=symbols('nXre nYre nZre',real=True)
nXim,nYim,nZim=[0,0,0]
n=Matrix([nXre+I*nXim,nYre+I*nYim,nZre+I*nZim])
# mbar
mXre,mYre,mZre=symbols('mXre mYre mZre',real=True)
mXim,mYim,mZim=symbols('mXim mYim mZim',real=True)
mbar=Matrix([mXre+I*mXim,mYre+I*mYim,mZre+I*mZim])
# constants
m0,n0=0,1/sqrt(2)

0 0 0


### 3+1 derivation of $\psi_4$

The Newman–Penrose scalar

$
\psi_4 = C_{\alpha\beta\gamma\delta}, n^\alpha \bar m^\beta n^\gamma \bar m^\delta
$
is evaluated by expressing the Weyl tensor in terms of 3+1 geometric quantities on a spatial slice.

We choose a null tetrad adapted to the foliation, with $m^\mu$ purely spatial ($m^0=0$). 

As a result, only the Weyl components

$
C_{ijkl}, \quad C_{0jkl}, \quad C_{0j0l}
$

contribute. 

Using Weyl symmetries, the contraction naturally splits into three terms involving

$
n^i\bar m^j n^k\bar m^l,\qquad
n_{[0}\bar m_{j]} n^k\bar m^l,\qquad
n_{[0}\bar m_{j]} n_{[0}\bar m_{l]} ,
$

with numerical factors arising from antisymmetrization and index permutations.

The required 4D Riemann components are rewritten using standard 3+1 identities:

* **Gauss equation**
  
  $
  {}^{(4)}R_{ijkl}
  = R_{ijkl} + 2K_{i[k}K_{l]j},
  $

  giving the purely spatial contribution.

* **Codazzi equation**
  
  $
  {}^{(4)}R_{0jkl}
  = 2D_{[k}K_{l]j}
  = 2\left(\partial_{[k}K_{l]j} + \Gamma^p_{j[k}K_{l]p}\right),
  $

  which produces the mixed time–space term.

* **Double-normal projection**

  Using standard ADM relations (and eliminating lapse/acceleration terms via gauge choice or evolution equations),
  
  $
  {}^{(4)}R_{0j0l}
  = R_{jl} - K_{jp}K^p_l + K K_{jl}.
  $

Combining these contributions yields the final 3+1 expression (see also Eq.5.1 of https://arxiv.org/pdf/gr-qc/0104063):

$\psi _4 = \left[ {R}_{ijkl}+2K_{i[k}K_{l]j}\right]
{n}^i\bar{m}^j{n}^k\bar{m}^l
-8\left[ K_{j[k,l]}+{\Gamma }_{j[k}^pK_{l]p}\right]
{n}^{[0}\bar{m}^{j]}{n}^k\bar{m}^l 
+4\left[ {R}_{jl}-K_{jp}K_l^p+KK_{jl}\right]
{n}^{[0}\bar{m}^{j]}{n}^{[0}\bar{m}^{l]}$

In the following we will construct the individual terms separately and extract the real and imaginary parts of $\psi_4$.

In [4]:
# useful to lower index in n
nd=sp.MutableDenseNDimArray.zeros(dim)
for i in range(dim):
    for j in range(dim):
        nd[i]+=gdd[i,j]*n[j]
# 1st term
p1=sum(
    Ruddd[i,j,k,l]*nd[i]*mbar[j]*n[k]*mbar[l]
    for i in range(dim)
    for j in range(dim)
    for k in range(dim)
    for l in range(dim)
)
# 2nd term
p2=sum(
    (Kdd[i,k]*Kdd[l,j]-Kdd[i,l]*Kdd[k,j])*n[i]*mbar[j]*n[k]*mbar[l]
    for i in range(dim)
    for j in range(dim)
    for k in range(dim)
    for l in range(dim)
)
# 3rd term
p3=sum(
    -2*(diff(Kdd[j,k],xu[l])-diff(Kdd[j,l],xu[k]))*(n0*mbar[j]-n[j]*m0)*n[k]*mbar[l]
    for j in range(dim)
    for k in range(dim)
    for l in range(dim)
)
# 4th term
p4=sum(
    -2*(Gammaudd[p,j,k]*Kdd[l,p]-Gammaudd[p,j,l]*Kdd[k,p])*(n0*mbar[j]-n[j]*m0)*n[k]*mbar[l]
    for j in range(dim)
    for k in range(dim)
    for l in range(dim)
    for p in range(dim)
)
# 5th term
p5=sum(
    Rdd[j,l]*(n0*mbar[j]-n[j]*m0)*(n0*mbar[l]-n[l]*m0)
    for j in range(dim)
    for l in range(dim)
)
# raising one index on K
Kud=sp.MutableDenseNDimArray.zeros(dim, dim)
for i in range(dim):
    for j in range(dim):
        for k in range(dim):
            Kud[i,j]+=guu[i,k]*Kdd[k,j]
        Kud[i,j]=simplify(Kud[i,j])
# 6th term
p6=sum(
    Kdd[j,p]*Kud[p, l]*(n0*mbar[j]-n[j]*m0)*(n0*mbar[l]-n[l]*m0)
    for j in range(dim)
    for l in range(dim)
    for p in range(dim)
)
# 7th term
p7=sum(
    Khat*Kdd[j,l]*(n0*mbar[j]-n[j]*m0)*(n0*mbar[l]-n[l]*m0)
    for j in range(dim)
    for l in range(dim)
)
# psi4
psi4sum=p1+p2+p3+p4+p5+p6+p7

Replacement rules that convert spatial derivatives of fields into independent symbolic variables.

Necessary to identify real and imaginary parts later on, since sympy can't deduce that derivatives of real functions are also real.

Also code generator assumes symbols and not functions.

In [5]:
# replacement rules for derivatives and turning functions into symbols
repl = {
    **{diff(Function("chi",real=True)(*xu),xu[i]): symbols(f"dchi_dx[{i}]",real=True) for i in range(3)},
    **{diff(Function("chi", real=True)(*xu), xu[j], xu[k]): symbols(f"ddchi_dx2[{j*(j+1)//2 + k}]", real=True) for j in range(3) for k in range(j + 1)},
    **{diff(Function("Khat",real=True)(*xu),xu[i]): symbols(f"dKhat_dx[{i}]",real=True) for i in range(3)},
    **{diff(Function(f"gtdd[{i}]",real=True)(*xu),xu[j]): symbols(f"dgtdd_dx[{j+3*i}]",real=True) for i in range(6) for j in range(3)},
    **{diff(Function(f"Atdd[{i}]",real=True)(*xu),xu[j]): symbols(f"dAtdd_dx[{j+3*i}]",real=True) for i in range(6) for j in range(3)},
    **{diff(Function(f"gtdd[{i}]",real=True)(*xu),xu[j],xu[k]): symbols(f"ddgtdd_dx2[{6*i + j*(j+1)//2 + k}]",real=True) for i in range(6) for j in range(3) for k in range(j + 1)},
    Function("chi",real=True)(*xu): symbols("chi",real=True),
    Function("Khat",real=True)(*xu): symbols("Khat",real=True),
    **{Function(f"gtdd[{i}]",real=True)(*xu): symbols(f"gtdd[{i}]",real=True) for i in range(6)},
    **{Function(f"Atdd[{i}]",real=True)(*xu): symbols(f"Atdd[{i}]",real=True) for i in range(6)}
}

Summing individual terms of (complex) $\psi_4$ and then turning functions into symbols is the slowest step (~3 minutes on a laptop). 

In [6]:
# full complex expression for psi4
psi4=psi4sum.subs(repl)

Replacement rules for the explicit definition of the tetrad.

In [12]:
# replacement for tetrad components
repltetrad={
    nXre: (-eRadial[0]/sqrt(2)).subs(repl),
    nYre: (-eRadial[1]/sqrt(2)).subs(repl),
    nZre: (-eRadial[2]/sqrt(2)).subs(repl),
    nXim: 0,
    nYim: 0,
    nZim: 0,
    mXre: (ePolar[0]/sqrt(2)).subs(repl),
    mYre: (ePolar[1]/sqrt(2)).subs(repl),
    mZre: (ePolar[2]/sqrt(2)).subs(repl),
    mXim: (eAzimuthal[0]/sqrt(2)).subs(repl),
    mYim: (eAzimuthal[1]/sqrt(2)).subs(repl),
    mZim: (eAzimuthal[2]/sqrt(2)).subs(repl)
}

Extracting real and imaginary part of full expression for $\psi_4$ first and then replacing the explicit definition of the tetrad components (doing it in this order turns out to be rather quick).

In [13]:
# real and imaginary part of psi4
psi4re=re(psi4).subs(repltetrad)
psi4im=im(psi4).subs(repltetrad)

Here the file "psi4_subexpressions.hh" used by the grace code is written (~2 minutes).

In [14]:
# Write some code. ABI 
scalar=("double",None)
tens=("double", [6])
vec=("double",[3])
ABI={
    "xx1": scalar,
    "xx2": scalar,
    "xx3": scalar,

    "gtdd": tens,
    "dgtdd_dx": ("double", [6*3]),
    "ddgtdd_dx2": ("double", [6*6]),
    
    "Atdd": tens,
    "dAtdd_dx": ("double", [6*3]),
    
    "chi": scalar,
    "dchi_dx": vec,
    "ddchi_dx2": ("double", [6]),
    
    "Khat": scalar,
    "dKhat_dx": vec
}

# expression to build
exprs=[psi4re,psi4im]

out_abi = {"psi4re": scalar, "psi4im": scalar}
out_list = ["psi4re", "psi4im"]

flist = []
flist.append(make_function(exprs,printer,"get_psi4",ABI,out_list,out_abi))

printed_functions = '\n'+'\n'.join(flist)

file = '''
/****************************************************************************/
/*                       psi4 helpers, SymPy generated                       */
/****************************************************************************/
#ifndef GRACE_PSI4_SUBEXPR_HH
#define GRACE_PSI4_SUBEXPR_HH

#include <Kokkos_Core.hpp>
''' + printed_functions + '''
#endif 
'''

with open("psi4_subexpressions.hh","w") as ff:
    ff.write(file)